# Astra — word-level English base + chat fine-tune (Kaggle GPU)

Trains the 8.1M-param word-level model on a Kaggle GPU (T4/P100) in roughly 15–25 min:
1. **word-prose** 5400 steps (English base, val ppl ~56–58)
2. **word-chat** 3800 steps warm-started from prose final (chat fine-tune)

**Persistence** (better than Colab): commit a version and the outputs in `/kaggle/working/output` are saved to the version's **Output** tab — no VM-recycle wipe. You can also push them to a Kaggle Dataset via the API (cell 13).

Runtime setup: **Settings → Accelerator: GPU (T4)**, Internet: ON (required for the git clone).

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

In [ ]:
import os, subprocess
REPO = '/kaggle/working/astra'
if os.path.isdir(REPO):
    subprocess.run(f'rm -rf {REPO}', shell=True, check=True)
subprocess.run(f'git clone --depth 1 https://github.com/anoneurx/astra.git {REPO}', shell=True, check=True)
assert os.path.isdir(REPO), 'clone failed - check Internet is ON and retry'
os.chdir(REPO)
print('cwd:', os.getcwd())

**Build the word tokenizer artifact** (gitignored, so generated here from the corpora):

In [ ]:
import os
if not os.path.exists('tokenizer/artifacts/prose_chat_word.json'):
    !python tools/build_word_tokenizer.py
import json
a = json.load(open('tokenizer/artifacts/prose_chat_word.json'))
print('tokenizer vocab:', a['vocab_size'])

**Smoke test** — 10 steps of word-prose to verify the pipeline on GPU before the real run.

In [ ]:
import os, subprocess
if not os.path.isdir('/kaggle/working/astra'):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git /kaggle/working/astra', shell=True, check=True)
os.chdir('/kaggle/working/astra')
if not os.path.exists('tokenizer/artifacts/prose_chat_word.json'):
    subprocess.run('python tools/build_word_tokenizer.py', shell=True, check=True)
!mkdir -p /kaggle/working/runs
!python training/gpu_train.py --config configs/astra5m_word_prose.json \
  --steps 10 --out /kaggle/working/runs/word_prose_smoke \
  --cache-dir /kaggle/working/cache 2>&1 | tail -8

Sanity: last line should show `[torch] device=cuda`, a `[step ...]` line, and `checkpoint -> ...final.npz`. If a step pinned here, check corpus cache in `/kaggle/working/cache`.

---
**1/2 — word-prose base (5400 steps)**

Look for `[step 5400]` and `checkpoint -> /kaggle/working/runs/word_prose/final.npz`.

In [ ]:
import os, subprocess
if not os.path.isdir('/kaggle/working/astra'):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git /kaggle/working/astra', shell=True, check=True)
os.chdir('/kaggle/working/astra')
if not os.path.exists('tokenizer/artifacts/prose_chat_word.json'):
    subprocess.run('python tools/build_word_tokenizer.py', shell=True, check=True)
!python training/gpu_train.py --config configs/astra5m_word_prose.json \
  --steps 5400 --out /kaggle/working/runs/word_prose \
  --cache-dir /kaggle/working/cache 2>&1 | tail -20
print('=== prose final ===')
!ls -la /kaggle/working/runs/word_prose/final.npz

**2/2 — word-chat fine-tune (3800 steps, warm-started from prose final)**

Look for `[step 3800]` and `checkpoint -> /kaggle/working/runs/word_chat/resumed/final.npz`.

In [ ]:
import os, subprocess
if not os.path.isdir('/kaggle/working/astra'):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git /kaggle/working/astra', shell=True, check=True)
os.chdir('/kaggle/working/astra')
if not os.path.exists('tokenizer/artifacts/prose_chat_word.json'):
    subprocess.run('python tools/build_word_tokenizer.py', shell=True, check=True)
!python training/gpu_train.py --config configs/astra5m_word_chat.json \
  --steps 3800 --out /kaggle/working/runs/word_chat \
  --resume /kaggle/working/runs/word_prose/final.npz --reset-step \
  --cache-dir /kaggle/working/cache 2>&1 | tail -20
print('=== chat final ===')
!ls -la /kaggle/working/runs/word_chat/resumed/final.npz

**Stage outputs** — everything under `/kaggle/working/output` is automatically saved as the version's **Output** tab (persistent, downloadable from the Kaggle UI):

In [ ]:
import os, shutil
out = '/kaggle/working/output'
os.makedirs(out, exist_ok=True)
pairs = [
    ('/kaggle/working/runs/word_prose/final.npz', 'word_prose_final.npz'),
    ('/kaggle/working/runs/word_chat/resumed/final.npz', 'word_chat_final.npz'),
]
for src, dst in pairs:
    if os.path.exists(src):
        shutil.copy(src, os.path.join(out, dst))
        print(f'[ok] {dst} ({os.path.getsize(os.path.join(out, dst)):,} bytes)')
    else:
        print(f'MISSING: {src}')
print('\nDownload: open this notebook -> Commit & Run All (Save version) -> Output tab.')
print('Or grab files now from the Output panel / file browser: /kaggle/working/output')

**Optional: push to a Kaggle Dataset via API** (most durable — survives everything). Requires the kernel secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` (Settings -> Secrets -> Add). Creates `astra-word-checkpoints` on first run, versions it afterwards.

Then on your local machine: `kaggle datasets download -d <username>/astra-word-checkpoints`.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = s.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = s.get_secret('KAGGLE_KEY')
except Exception as e:
    print('no Kaggle API secrets configured - skip or add KAGGLE_USERNAME/KAGGLE_KEY')
else:
    import json
    out = '/kaggle/working/output'
    meta = {
        'id': f"{os.environ['KAGGLE_USERNAME'].lower()}/astra-word-checkpoints",
        'title': 'Astra word checkpoints',
        'licenses': [{'name': 'MIT'}],
    }
    with open(os.path.join(out, 'dataset-metadata.json'), 'w') as f:
        json.dump(meta, f)
    r = os.system(f'kaggle datasets create -p {out}')
    if r != 0:
        os.system(f'kaggle datasets version -p {out} -m "checkpoints"')